# 8.1 FGSM 공격

정상 이미지에 노이즈를 더해 머신러닝 모델을 헷갈리게 하는 이미지가
바로 적대적 예제(Adversarial Example) 입니다.
이 프로젝트에선 Fast Gradient Sign Method, 즉 줄여서 FGSM이라는 방식으로
적대적 예제를 생성해 미리 학습이 완료된 딥러닝 모델을 공격해보도록 하겠습니다.

In [1]:
import torch # 파이토치 라이브러리 임포트
import torch.nn.functional as F # 신경망 기능 함수 모듈 임포트
import torchvision.models as models # 사전 학습된 모델 모듈 임포트
import torchvision.transforms as transforms # 이미지 변환 도구 임포트

from PIL import Image # 이미지 처리 라이브러리
import json # JSON 처리 모듈

In [2]:
import matplotlib.pyplot as plt # 시각화 도구 임포트

## 학습된 모델 불러오기

In [3]:
model = models.resnet101(pretrained=True) # 사전 학습된 ResNet101 모델 로드
model.eval() # 모델을 평가 모드로 설정
print(model) # 모델 구조 출력

## 데이터셋 불러오기

In [4]:
CLASSES = json.load(open('./imagenet_samples/imagenet_classes.json')) # 클래스 이름 JSON 로드
idx2class = [CLASSES[str(i)] for i in range(1000)] # 인덱스별 클래스 이름 리스트 생성

## 이미지 불러오기

In [5]:
# 공격 대상 이미지 파일 열기
img = Image.open('imagenet_samples/corgie.jpg')

In [6]:
# 이미지를 텐서로 변환하는 파이프라인
img_transforms = transforms.Compose([
    transforms.Resize((224, 224), Image.BICUBIC), # 크기 조정
    transforms.ToTensor(), # 텐서 변환
])

img_tensor = img_transforms(img) # 이미지 전처리 적용
img_tensor = img_tensor.unsqueeze(0) # 배치 차원 추가

print("이미지 텐서 모양:", img_tensor.size()) # 크기 확인

In [7]:
# 시각화를 위해 텐서를 넘파이 행렬로 변환
original_img_view = img_tensor.squeeze(0).detach()  # 배치 차원 제거 및 분리
original_img_view = original_img_view.transpose(0,2).transpose(0,1).numpy() # (H, W, C)로 변환

# 원본 이미지 출력
plt.imshow(original_img_view)

## 공격 전 성능 확인하기


In [8]:
output = model(img_tensor) # 모델 예측
prediction = output.max(1, keepdim=False)[1] # 가장 높은 확률의 클래스 인덱스

prediction_idx = prediction.item() # 정수형 인덱스
prediction_name = idx2class[prediction_idx] # 클래스 이름

print("예측된 레이블 번호:", prediction_idx) # 결과 출력
print("레이블 이름:", prediction_name)

## FGSM 공격 함수 정의

In [9]:
def fgsm_attack(image, epsilon, gradient):
    # 기울기값의 원소의 sign(부호) 값을 구함
    sign_gradient = gradient.sign()
    # 이미지 각 픽셀의 값을 sign_gradient 방향으로 epsilon만큼 조절
    perturbed_image = image + epsilon * sign_gradient
    # [0,1] 범위를 벗어나는 값을 조절 (클리핑)
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image # 생성된 적대적 예제 반환

## 적대적 예제 생성

In [10]:
# 이미지의 기울기값을 구하도록 설정
img_tensor.requires_grad_(True)

# 이미지를 모델에 통과시켜 출력값 계산
output = model(img_tensor)

# 오차값 계산 (레이블 263은 웰시코기)
loss = F.nll_loss(output, torch.tensor([263])) 

# 모델 가중치의 기울기를 초기화하고 역전파 수행
model.zero_grad()
loss.backward()

# 입력 이미지의 기울기값 추출
gradient = img_tensor.grad.data

# FGSM 공격으로 적대적 예제 생성
epsilon = 0.03 # 공격 강도
perturbed_data = fgsm_attack(img_tensor, epsilon, gradient)

# 생성된 적대적 예제를 다시 모델에 통과시킴
output = model(perturbed_data)

## 적대적 예제 성능 확인

In [11]:
perturbed_prediction = output.max(1, keepdim=True)[1] # 공격 후 예측값 추출

perturbed_prediction_idx = perturbed_prediction.item() # 정수형 인덱스
perturbed_prediction_name = idx2class[perturbed_prediction_idx] # 공격 후 클래스 이름

print("예측된 레이블 번호:", perturbed_prediction_idx) # 결과 출력
print("레이블 이름:", perturbed_prediction_name)

In [12]:
# 적대적 예제 시각화를 위해 넘파이로 변환
perturbed_data_view = perturbed_data.squeeze(0).detach()
perturbed_data_view = perturbed_data_view.transpose(0,2).transpose(0,1).numpy()

# 공격받은 이미지 출력
plt.imshow(perturbed_data_view)

## 원본과 적대적 예제 비교

In [13]:
f, a = plt.subplots(1, 2, figsize=(10, 10)) # 비교를 위한 서브플롯 생성

# 원본 이미지 표시
a[0].set_title(prediction_name)
a[0].imshow(original_img_view)

# 적대적 예제 표시
a[1].set_title(perturbed_prediction_name)
a[1].imshow(perturbed_data_view)

plt.show() # 결과 시각화